# Module 2 — Context Managers

**What it is:** Context managers guarantee setup and teardown around a block of code using the `with` statement. They implement `__enter__` and `__exit__`, or use `@contextmanager` from `contextlib`.

**Where we use it:** File I/O, database connections, locks, temporary directories, and any resource that must be released reliably.

**Why we use it:** They prevent resource leaks and make cleanup automatic even when exceptions occur.

## Built-in file context manager

In [ ]:
from pathlib import Path

OUTPUT = Path("Module2/output")
OUTPUT.mkdir(parents=True, exist_ok=True)
path = OUTPUT / "context_demo.txt"

# with ensures the file is closed even if an error happens inside the block.
with path.open("w", encoding="utf-8") as handle:
    handle.write("Written inside a with block\n")

print(path.read_text(encoding="utf-8"))

## `@contextmanager` decorator

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label="Block"):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label} finished in {elapsed:.4f}s")

with timer("Sum loop"):
    total = sum(range(1_000_000))
    print("Total:", total)

## Class-based context manager

In [ ]:
# Implement __enter__ and __exit__ for reusable resource management.

class ManagedFile:
    def __init__(self, path, mode):
        self.path = path
        self.mode = mode
        self.handle = None

    def __enter__(self):
        self.handle = open(self.path, self.mode, encoding="utf-8")
        return self.handle

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.handle:
            self.handle.close()
        # Return False to propagate exceptions; True would suppress them.
        return False

from pathlib import Path
path = Path("Module2/output/managed.txt")
path.parent.mkdir(parents=True, exist_ok=True)

with ManagedFile(path, "w") as f:
    f.write("Managed by a class-based context manager\n")

print(path.read_text(encoding="utf-8"))

## `contextlib.suppress` and nested contexts

In [ ]:
from contextlib import suppress, ExitStack
from pathlib import Path

# suppress ignores specific exceptions inside the block.
with suppress(FileNotFoundError):
    Path("Module2/output/does_not_exist.txt").read_text()

# ExitStack manages multiple context managers dynamically.
OUTPUT = Path("Module2/output")
OUTPUT.mkdir(parents=True, exist_ok=True)

with ExitStack() as stack:
    f1 = stack.enter_context((OUTPUT / "a.txt").open("w", encoding="utf-8"))
    f2 = stack.enter_context((OUTPUT / "b.txt").open("w", encoding="utf-8"))
    f1.write("File A\n")
    f2.write("File B\n")

print("Created:", list(OUTPUT.glob("*.txt")))